In [1]:
!pip install pyserial

In [2]:
#Import necessary libraries
import os
import serial
import time
import threading
import struct
import queue
import tkinter as tk
from tkinter import simpledialog
import numpy as np
import tensorflow as tf

#load the test data

SCRIPT_DIR = os.getcwd()
DATA_DIR   = os.path.join(SCRIPT_DIR, "processed")


X_test      = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_test      = np.load(os.path.join(DATA_DIR, "y_test.npy"))

#remove overlap for X_test
X_test = X_test[::3]

y_test = np.where(y_test > 0, 1, y_test)
X_test = X_test.flatten()

In [3]:
#get the values needed to quantize before sending to Arduino
interpreter = tf.lite.Interpreter(model_path='model.tflite')
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

#get parameters to convert the floating point data into int8
in_scale, in_zp = input_details['quantization']

C:\Users\Thad\anaconda3\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [4]:


#Variable needed to connect to Arduino board, serialport may change when connection lost
serialport = 'COM6'
baudrate = 115200

#frequency and timing information
frequency = 360.0
duration = 1
#counter keeps track of window count, purposefully picked a starting value of interest
counter = 4628

# threading event created to signal to use when arduino signals for more data
trigger = threading.Event()

# queue used to send box updates from the worker threads to Demo GUI
status_queue = queue.Queue()

#code to put things in queue to update UI
def set_status(key, text, color="black"):
    #Push an update to GUI
    status_queue.put({"key": key, "text": text, "color": color})


#function to monitor output from Arduino
def read(ser):
    print("Listening for Arduino signals")
    
    last_k_time = None
    heap= None
    stack = None
    arena = None
    while True:
        try:
            if ser.in_waiting > 0:
                incomingdata = ser.readline().decode('utf-8', errors='ignore').strip()

                if incomingdata:
                    print(f"Arduino Output: {incomingdata}")
                    set_status("Latest Output", incomingdata)

                    #Watch for the 'k' ready signal, start output if signal received
                    if incomingdata.lower() == 'k':
                        now = time.time()
                        if last_k_time is not None:
                            interval_ms = (now - last_k_time) * 1000
                            print(f"[TIMING] Interval since last 'k': {interval_ms:.1f} ms")
                            
                        last_k_time = now
                        trigger.set()

                    #read various other outputs and send to UI
                    elif incomingdata.startswith("Cl"):  
                        prediction = incomingdata
                        label_value = y_test[counter]
                        truelabel = "Normal" if label_value == 0 else "Abnormal"
                        set_status("Prediction", f"{prediction} \n True Label: {truelabel}")

                    elif incomingdata.startswith("In"):
                        latency=incomingdata
                        set_status("Timing", f"Data Request Interval:{interval_ms:.1f} ms\n {incomingdata}")

                    elif incomingdata.startswith("He"):
                        heap=incomingdata

                    elif incomingdata.startswith("St"):
                        stack=incomingdata

                    elif incomingdata.startswith("Ar"):
                        arena=incomingdata
                        set_status("Last RAM Update", f"{heap} \n {stack} \n {arena}")
                

        #breaks loop if error message received
        except Exception as e:
            print(f"Error reading: {e}")
            
            break
        time.sleep(0.001)

#write loop, waits for signal, then sends data for 1 second after signal received
def write(ser):
    global counter
    interval = 1.0 / frequency



    #Tracker for first loop, need to send 3 seconds the first time to fill up the buffer
    firstrun = True

    #tracker for index of sent data
    data_index = 1666800;

    #continuous loop for output
    while True:
        #wait until trigger
        trigger.wait()

        #fill buffer on first run, otherwise just send data for duration
        shift = 3.0 if firstrun else duration

        #calculate timing of data sending
        starttime = time.time()
        endtime = starttime + shift
        nexttime = starttime

        print("Sending Data")

        try:
            while time.time() < endtime:
                currenttime = time.time()

                # Check if it is time to send the next packet based on the frequency
                if currenttime >= nexttime:
                    value=X_test[data_index]

                    #quantize data using scale used to quantize model
                    quantized = int(round(value / in_scale + in_zp))
                    quantized = max(-128, min(quantized, 127))  

                    payload = struct.pack('<b', quantized)  # 1 byte, signed

                    # Send the data
                    ser.write(payload)

                    data_index += 1

                    # Schedule the next packet time
                    nexttime += interval

                time.sleep(0.001)

        except Exception as e:
            print(f"Error sending: {e}")
            
            break

        #Clear event to wait for next signal
        firstrun = False
        trigger.clear()
        counter +=1
        


#function to connect to arduino and start threads
def connect_and_start():
    print(f"Connecting to Arduino on {serialport}...")
    try:
        # Open port
        ser = serial.Serial(serialport, baudrate, timeout=0.1)

        # Clear out any noise in the buffers right at connection time
        ser.reset_input_buffer()
        ser.reset_output_buffer()
        print("Connected.")
        

    except Exception as e:
        print(f"Could not open port: {e}")
        
        return

    # start read and write threads
    reader = threading.Thread(target=read, args=(ser,), daemon=True)
    writer = threading.Thread(target=write, args=(ser,), daemon=True)

    reader.start()
    writer.start()


#class to create simple UI
class Dashboard:
    def __init__(self, root):
        self.root = root
        self.root.title("Demo")
        self.root.geometry("1200x800")

        # container that holds all the boxes in a grid
        self.grid_frame = tk.Frame(root)
        self.grid_frame.pack(expand=True, fill="both", padx=10, pady=10)

        self.boxes = {}   
        self.next_col = 0
        self.next_row = 0
        self.cols_per_row = 2

        # creating the boxes
        for key in ("Latest Output", "Last RAM Update",
                    "Timing","Prediction"):
            self.add_box(key)

        

        self.poll_queue()

    def add_box(self, key, initial_text="--"):
        if key in self.boxes:
            return

        frame = tk.Frame(self.grid_frame, relief="groove", borderwidth=2)
        frame.grid(row=self.next_row, column=self.next_col, sticky="nsew", padx=6, pady=6)
        self.grid_frame.grid_columnconfigure(self.next_col, weight=1)
        self.grid_frame.grid_rowconfigure(self.next_row, weight=1)

        title_label = tk.Label(frame, text=key, font=("Segoe UI", 11, "bold"))
        title_label.pack(pady=(8, 2))

        if key == "Latest Output":
    # container so the text scrolls
            text_frame = tk.Frame(frame)
            text_frame.pack(expand=True, fill="both", padx=10, pady=(0, 10))

            scrollbar = tk.Scrollbar(text_frame)
            scrollbar.pack(side="right", fill="y")

            text_widget = tk.Text(text_frame, height=6, wrap="word",
                               font=("Segoe UI", 11), yscrollcommand=scrollbar.set)
            text_widget.pack(side="left", expand=True, fill="both")
            scrollbar.config(command=text_widget.yview)

            text_widget.insert("end", initial_text)
            text_widget.config(state="disabled")  # read-only, user can still scroll/select

            self.boxes[key] = {"frame": frame, "value_label": text_widget, "is_text": True}
        
        #every other box
        else:
            value_label = tk.Label(frame, text=initial_text, font=("Segoe UI", 18),
                                wraplength=400, justify="center")
            value_label.pack(expand=True, fill="both", padx=10, pady=(0, 10))
            self.boxes[key] = {"frame": frame, "value_label": value_label, "is_text": False}

        self.next_col += 1
        if self.next_col >= self.cols_per_row:
            self.next_col = 0
            self.next_row += 1


    #update screen, using a queue prevents UI crashes
    def poll_queue(self):
        try:
            while True:
                item = status_queue.get_nowait()
                key, text, color = item["key"], item["text"], item.get("color", "black")
                if key not in self.boxes:
                    self.add_box(key)

                box = self.boxes[key]
                if box.get("is_text"):
                    widget = box["value_label"]
                    widget.config(state="normal")
                    widget.insert("end", text + "\n")
                    widget.see("end")        
                    widget.config(state="disabled")
                else:
                    box["value_label"].config(text=text, fg=color)
        except queue.Empty:
            pass
        self.root.after(50, self.poll_queue)


def main():
    root = tk.Tk()
    Dashboard(root)

    # kick off serial connection + threads once the window is up
    root.after(100, connect_and_start)

    root.mainloop()


if __name__ == "__main__":
    main()


Connecting to Arduino on COM6...
Connected.
Listening for Arduino signals
Arduino Output: Model loaded
Arduino Output: Arena used: 56772 / 92160 bytes
Arduino Output: k
Sending Data
Arduino Output: k
[TIMING] Interval since last 'k': 3005.3 ms
Sending Data
Arduino Output: Result: 0.9219, 0.0781
Arduino Output: Classification: Normal
Arduino Output: Invoke time (ms): 1427
Arduino Output: k
[TIMING] Interval since last 'k': 1439.0 ms
Sending Data
Arduino Output: Result: 0.9609, 0.0391
Arduino Output: Classification: Normal
Arduino Output: Invoke time (ms): 1427
Arduino Output: k
[TIMING] Interval since last 'k': 1423.8 ms
Sending Data
Arduino Output: Result: 0.9570, 0.0430
Arduino Output: Classification: Normal
Arduino Output: Invoke time (ms): 1426
Arduino Output: k
[TIMING] Interval since last 'k': 1433.9 ms
Sending Data
Arduino Output: Result: 0.9727, 0.0273
Arduino Output: Classification: Normal
Arduino Output: Invoke time (ms): 1427
Arduino Output: k
[TIMING] Interval since last 'k'